In [ ]:
import os
import subprocess
from pathlib import Path
from zipfile import ZipFile

import pandas as pd

from paths import paths, prepare_preprocessing

In [ ]:
prepare_preprocessing()

In [ ]:
SELECTED_MODE = 'bus' # 'bus', 'rail', or 'tfl'
GTFS = paths.preprocessing.transport.gtfs[SELECTED_MODE]

In [ ]:
# Convert rail and TfL source feeds when their generated GTFS archive is absent
if SELECTED_MODE in ('rail', 'tfl') and not GTFS.is_file():
    raw = paths.raw.walkability.atoc if SELECTED_MODE == 'rail' else paths.raw.walkability.transxchange
    
    script = """
    library(UK2GTFS)
    args <- commandArgs(trailingOnly = TRUE)
    gtfs <- if (args[1] == "rail") {
        nr2gtfs(path_in = args[2], ncores = 6)
    } else {
        transxchange2gtfs(path_in = args[2], ncores = 6)
    }
    gtfs_write(gtfs, folder = args[3], name = args[4])
    """
    
    subprocess.run(
        [
            'Rscript',
            '--vanilla',
            '-e',
            script,
            SELECTED_MODE,
            str(raw),
            str(paths.preprocessing.transport.directory),
            GTFS.stem
        ],
        check = True,
    )

In [ ]:
# Extract and load the GTFS data
def unzip_gtfs(zip_path: Path):
    with ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(FEED.directory)

FEED = paths.preprocessing.transport.feeds[SELECTED_MODE]
unzip_gtfs(GTFS)

stops = pd.read_csv(FEED.stops)
stop_times = pd.read_csv(FEED.stop_times)
trips = pd.read_csv(FEED.trips)
calendar = pd.read_csv(FEED.calendar)
cal_dates = pd.read_csv(FEED.calendar_dates)

In [ ]:
# Init empty service day mapping
day_cols = ['monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday']

service_day = pd.DataFrame(columns = ['service_id', 'weekday'])

if not calendar.empty and calendar[day_cols].to_numpy().sum():
    service_day = (
        calendar[['service_id', *day_cols]]
        .melt(id_vars = 'service_id', var_name = 'weekday', value_name = 'runs')
        .query('runs == 1')[['service_id', 'weekday']]
    )

In [ ]:
# Parse exception dates and separate added vs. removed services
cal_dates = cal_dates.assign(weekday = pd.to_datetime(cal_dates['date'], format = '%Y%m%d').dt.day_name().str.lower())

added = cal_dates.query('exception_type == 1')[['service_id', 'weekday']]
removed = cal_dates.query('exception_type == 2')[['service_id', 'weekday']]

In [ ]:
# Combine base schedule with exceptions, applying additions and removals to get active service day pairs
service_day = (
    pd.concat([service_day.assign(flag = 1), added.assign(flag = 1), removed.assign(flag = -1)], ignore_index = True)
    .groupby(['service_id', 'weekday'], as_index = False)['flag']
    .sum()
    .query('flag > 0')
    .drop('flag', axis = 1)
)

In [ ]:
# Select stop time fields, merge with trip definitions, and join servic -day to tag each stop with its active day
stop_times_reduced = stop_times[['trip_id', 'stop_id', 'departure_time']].copy()

trips_cols = ['trip_id', 'service_id', 'route_id']

if 'direction_id' in trips.columns:
    trips_cols.append('direction_id')

trip_stop = stop_times_reduced.merge(trips[trips_cols], on = 'trip_id').merge(service_day, on = 'service_id')

trip_stop = trip_stop.drop_duplicates(subset = ['stop_id', 'weekday', 'departure_time'])

In [ ]:
# Count services per stop per day, pivot into day columns, and compute the average daily frequency
services_per_stop_day = (
    trip_stop.groupby(['stop_id', 'weekday']).size().unstack(fill_value = 0).reindex(columns = day_cols, fill_value = 0)
)

services_per_stop_day['avg_per_day'] = services_per_stop_day.mean(axis = 1)

In [ ]:
# Reindex and merge in stop metadata
freq = services_per_stop_day.reset_index()
freq_with_meta = freq.merge(stops, on = 'stop_id', how = 'left')
cols = ['stop_id', 'stop_code', 'stop_name', 'stop_lat', 'stop_lon', 'avg_per_day', *day_cols]
freq_with_meta = freq_with_meta[cols]

In [ ]:
# Filter out the depots and other non-passenger locations in the rail data
if SELECTED_MODE == 'rail':
    freq_with_meta = freq_with_meta.dropna(subset = ['stop_code'])

In [ ]:
# Aggregate TfL data by station to remove direction and line suffixes
if SELECTED_MODE == 'tfl':
    # Remove direction/line suffix
    freq_with_meta['station_id'] = freq_with_meta['stop_id'].astype(str).str[:-1]

    day_cols = ['monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday']

    agg = {col: 'sum' for col in day_cols}
    agg.update({'stop_name': 'first', 'stop_lat': 'mean', 'stop_lon': 'mean'})

    stations = freq_with_meta.groupby('station_id', as_index = False).agg(agg)

    for col in day_cols:
        if (stations[col] == 0).any():
            stations[col] = stations[day_cols].mode(axis = 1)[0]

    stations['avg_per_day'] = stations[day_cols].sum(axis = 1) / 7

    cols = ['station_id', 'stop_name', 'stop_lat', 'stop_lon', 'avg_per_day'] + day_cols
    freq_with_meta = stations[cols].rename(columns = {'station_id': 'stop_id'})

In [ ]:
freq_with_meta = freq_with_meta.sort_values(by = 'avg_per_day', ascending = False)
freq_with_meta.to_csv(getattr(paths.preprocessing.transport, SELECTED_MODE), index = False)